# 07 — Hybrid Regime-Switching Market & Climate VaR

**This is the dissertation's central methodological contribution**, reproduced here as an open, reusable Python implementation (see `src/hybrid_var.py`).

**Concept:** Market VaR and Climate VaR operate on incompatible horizons — days versus decades. Rather than picking one, this model treats every simulated short-horizon return as coming from one of two regimes:

- **Baseline regime** (probability 1 − p): an ordinary short-horizon market return.
- **Climate shock regime** (probability p, default 15%): the baseline return *plus* a long-horizon NGFS climate shock, rescaled down to the short VaR horizon under a linear-attribution assumption.

This lets a single VaR number reflect both everyday market risk **and** the (relatively small but non-zero, at 15%) chance that transition/physical climate risk materialises within the holding period — the dissertation's own answer to 'how do you compare a 5-day VaR against a to-2050 climate scenario?'

**Important caveat (carried over honestly from the dissertation, Appendix D):** rescaling a 25-year shock down to 5 days by simple linear attribution is a simplification — real transition risk is likely non-linear and could arrive in a sudden repricing rather than smoothly. The hybrid model should be read as **illustrative of the direction and rough magnitude of the effect**, not a precise forecast.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.climate_var import estimate_capm_beta, climate_scenario_shock, SAMPLE_NGFS_SCENARIOS
from src.hybrid_var import run_hybrid_var_scenarios, rescale_climate_shock

returns = pd.read_csv('../data/market_prices.csv', index_col=0, parse_dates=True)
returns = np.log(returns / returns.shift(1)).dropna()


## 1. Reuse the CAPM beta and scenario shocks from notebook 06

In [ ]:
MARKET_TICKER = 'SPY'
ASSET_TICKER = 'USO'

asset_returns = returns[ASSET_TICKER]
market_returns = returns[MARKET_TICKER]

beta = estimate_capm_beta(asset_returns, market_returns, risk_free_rate=0.0)
scenario_shocks = {
    name: climate_scenario_shock(market_change, beta)
    for name, market_change in SAMPLE_NGFS_SCENARIOS.items()
}
scenario_shocks


## 2. Run the Hybrid VaR model across all scenarios

Reproduces the dissertation's Tables 18/19 (rescaled to a 5-day horizon, 15% climate regime probability — both adjustable below).

In [ ]:
hybrid_table = run_hybrid_var_scenarios(
    asset_returns,
    scenario_shocks,
    long_horizon_days=25 * 252,   # ~25 years to 2050, matching the dissertation
    target_horizon_days=5,
    climate_prob=0.15,
    confidence_levels=[0.95, 0.99],
    n_simulations=10_000,
)
hybrid_table.round(3)


## 3. Compare Hybrid VaR against plain Market VaR

The dissertation's key finding: Hybrid VaR values were **consistently worse** than Market VaR alone across every scenario — even a modest 15% climate-regime probability materially raises tail risk once it's actually embedded in the simulation, rather than sitting in a separate long-horizon report nobody reads day-to-day.

In [ ]:
from src.var import monte_carlo_var

market_var_95_5day = monte_carlo_var(asset_returns, confidence=0.95) * np.sqrt(5) * 100
market_var_99_5day = monte_carlo_var(asset_returns, confidence=0.99) * np.sqrt(5) * 100

print(f'Plain 5-day Market VaR  — 95%: {market_var_95_5day:.2f}%  |  99%: {market_var_99_5day:.2f}%')
print()
print('Hybrid VaR by scenario:')
hybrid_table[['hybrid_var_95_pct', 'hybrid_var_99_pct']].round(2)


## 4. Sensitivity check: does the climate-regime probability matter?

*(Re-run `run_hybrid_var_scenarios` at, say, climate_prob = 0.05, 0.15, 0.30 and plot how Hybrid VaR moves — a natural extension beyond the dissertation's single fixed 15% assumption, and a good chart for demonstrating this isn't a black box.)*

In [ ]:
# TODO: loop over climate_prob values and plot Hybrid VaR sensitivity


---

**This completes the reproduction of the dissertation's full methodology in open, testable Python.** Combined with notebooks 01–05, this project now covers: descriptive/stationarity analysis, four volatility models, three Market VaR methods, two backtests, historical/hypothetical stress scenarios, CAPM-beta Climate VaR under NGFS pathways, and the regime-switching Hybrid VaR — all runnable on live market data rather than a fixed academic submission.